# Kaggle CIFAR-100 Wasserstein geometry smoke test

This notebook orchestrates the existing repository; it does not duplicate training, model, feature extraction, or Wasserstein code. Before running, enable a Kaggle **GPU**, enable **Internet**, and attach a Kaggle secret named exactly `github_token`.

## 1. Kaggle environment and secure repository clone

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

import torch

assert torch.cuda.is_available(), "CUDA unavailable. Enable a GPU in Kaggle Notebook settings before continuing."
print("GPU count:", torch.cuda.device_count())
print("GPU name:", torch.cuda.get_device_name(0))
print("CUDA version:", torch.version.cuda)
print("PyTorch version:", torch.__version__)
print("Only cuda:0 is used by the existing single-device baseline.")

The variable is deliberately named `github_token` and is loaded from Kaggle Secrets. The token is exposed to Git only through a temporary `GIT_ASKPASS` process and is never embedded in the clone URL or printed.

In [ ]:
from kaggle_secrets import UserSecretsClient

try:
    github_token = UserSecretsClient().get_secret("github_token")
except Exception as exc:
    raise RuntimeError("Attach a Kaggle secret named 'github_token' to this notebook.") from exc
assert github_token, "Kaggle secret 'github_token' is empty"

REPO_URL = "https://github.com/duyh80456-code/new-pruning.git"
PROJECT_ROOT = Path("/kaggle/working/new-pruning")
askpass_path = Path("/kaggle/working/.github_git_askpass.py")
askpass_path.write_text(
    "#!/usr/bin/env python3\n"
    "import os, sys\n"
    "prompt = sys.argv[1] if len(sys.argv) > 1 else ''\n"
    "print('x-access-token' if 'Username' in prompt else os.environ['GITHUB_TOKEN_RUNTIME'])\n"
)
askpass_path.chmod(0o700)
git_environment = os.environ.copy()
git_environment.update({
    "GIT_ASKPASS": str(askpass_path),
    "GIT_TERMINAL_PROMPT": "0",
    "GITHUB_TOKEN_RUNTIME": github_token,
})
try:
    if (PROJECT_ROOT / ".git").is_dir():
        subprocess.run(["git", "-C", str(PROJECT_ROOT), "pull", "--ff-only"], env=git_environment, check=True)
    else:
        subprocess.run(["git", "clone", REPO_URL, str(PROJECT_ROOT)], env=git_environment, check=True)
finally:
    askpass_path.unlink(missing_ok=True)
    git_environment.pop("GITHUB_TOKEN_RUNTIME", None)
    github_token = None

assert (PROJECT_ROOT / "scripts" / "run_experiment.py").is_file()
assert (PROJECT_ROOT / "configs" / "kaggle_smoke.yaml").is_file(), "Push the Kaggle files to GitHub before running."
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
print("Repository ready at:", PROJECT_ROOT)

In [ ]:
# Kaggle already provides PyTorch/SciPy/pandas/sklearn/matplotlib. Install only the missing project profiler.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "thop>=0.1.1"], check=True)

## 2. Smoke-test configuration

One seed, 20 epochs, four training anchors, 16 evaluation widths, Sliced Wasserstein, and only the Euclidean-mean control.

In [ ]:
from datetime import datetime, timezone
import yaml

BASE_CONFIG_PATH = PROJECT_ROOT / "configs" / "kaggle_smoke.yaml"
config = yaml.safe_load(BASE_CONFIG_PATH.read_text())
anchors = [float(x) for x in config["compression"]["train_widths"]]
eval_widths = [float(x) for x in config["compression"]["eval_widths"]]
expected_grid = [round(0.25 + 0.05 * index, 2) for index in range(16)]
assert config["dataset"]["name"].lower() == "cifar100"
assert config["dataset"]["fake_data"] is False
assert config["experiment"]["seeds"] == [0]
assert config["training"]["epochs"] == 20
assert anchors == [0.25, 0.50, 0.75, 1.00]
assert eval_widths == expected_grid
assert config["geometry"]["method"] == "sliced_wasserstein"
assert config["geometry"]["control_methods"] == ["euclidean_mean"]
assert config["geometry"]["num_projections"] == 128
assert config["oracle"]["enabled"] is False

RUN_NAME = datetime.now(timezone.utc).strftime("kaggle-smoke-%Y%m%d-%H%M%S")
RUN_DIR = Path("/kaggle/working/new-pruning-outputs") / RUN_NAME
config["experiment"]["output_dir"] = str(RUN_DIR)
RESOLVED_CONFIG_PATH = Path("/kaggle/working/kaggle_smoke_resolved.yaml")
RESOLVED_CONFIG_PATH.write_text(yaml.safe_dump(config, sort_keys=False))
print(yaml.safe_dump(config, sort_keys=False))

## 3. Preflight and run the existing experiment entrypoint

The preflight downloads/opens real CIFAR-100 before GPU training. AMP remains disabled because the existing baseline has no AMP configuration.

In [ ]:
from data import build_loaders

print("GPU name:", torch.cuda.get_device_name(0))
print("CUDA version:", torch.version.cuda)
print("PyTorch version:", torch.__version__)
print("training widths:", anchors)
print("evaluation widths:", eval_widths)
print("seed: 0")
print("epochs: 20")
print("dataset mode: CIFAR-100, fake_data=False")
print("AMP: disabled (not exposed by the existing baseline)")

train_loader, val_loader, feature_loader = build_loaders(config, seed=0)
assert len(train_loader.dataset) == 50_000
assert len(val_loader.dataset) == 10_000
assert len(feature_loader.dataset) == 2_000
del train_loader, val_loader, feature_loader
print("CIFAR-100 preflight: OK")

In [ ]:
import time
from scripts.run_experiment import run

started = time.perf_counter()
base_report_path = run(RESOLVED_CONFIG_PATH)
elapsed_hours = (time.perf_counter() - started) / 3600
print(f"Experiment completed in {elapsed_hours:.2f} hours")
print("Base report:", base_report_path)

## 4. Training/evaluation status and integrity checks

In [ ]:
import json
import numpy as np
import pandas as pd

SEED_DIR = RUN_DIR / "seed_0"
RESULT_DIR = SEED_DIR / "results"
checkpoint = SEED_DIR / "checkpoint.pt"
assert checkpoint.is_file() and checkpoint.stat().st_size > 0, "Checkpoint was not saved"

training = pd.read_csv(SEED_DIR / "training_metrics.csv")
metrics = pd.read_csv(RESULT_DIR / "budget_metrics.csv").sort_values("budget").reset_index(drop=True)
local = pd.read_csv(RESULT_DIR / "local_sensitivity.csv")
control = pd.read_csv(RESULT_DIR / "control_metric_correlations.csv")
cliffs = pd.read_csv(RESULT_DIR / "compression_cliffs.csv")
wasserstein = np.load(RESULT_DIR / "wasserstein_matrix.npy")

assert set(training["width"].astype(float).unique()) == {0.25, 0.50, 0.75, 1.00}
assert len(metrics) == 16 and int(metrics["is_train_anchor"].sum()) == 4
assert np.isfinite(metrics[["accuracy", "loss", "flops", "params"]].to_numpy()).all()
assert np.all(np.diff(metrics["flops"].to_numpy(float)) > 0), "FLOPs do not rise monotonically with width"
assert wasserstein.shape == (16, 16) and np.isfinite(wasserstein).all()
assert np.allclose(wasserstein, wasserstein.T) and np.allclose(np.diag(wasserstein), 0)

feature_paths = sorted((SEED_DIR / "features").glob("features_budget_*.pt"))
assert len(feature_paths) == 16
reference_ids = None
for feature_path in feature_paths:
    payload = torch.load(feature_path, map_location="cpu", weights_only=False)
    assert torch.isfinite(payload["features"]).all()
    assert float(payload["features"].std()) > 1e-8, f"Degenerate features: {feature_path.name}"
    if reference_ids is None:
        reference_ids = payload["sample_ids"]
    else:
        assert torch.equal(reference_ids, payload["sample_ids"]), "Feature IDs/order differ across budgets"
print("All post-run integrity checks: OK")

## 5. Compact result table

In [ ]:
from IPython.display import Markdown, display

accuracy_table = metrics[["budget", "is_train_anchor", "accuracy", "flops", "params"]].copy()
accuracy_table["seen_unseen"] = np.where(accuracy_table["is_train_anchor"], "seen", "unseen")
display(accuracy_table[["budget", "seen_unseen", "accuracy", "flops", "params"]])

## 6. Accuracy vs budget

In [ ]:
import matplotlib.pyplot as plt

PLOT_DIR = RUN_DIR / "plots"
PLOT_DIR.mkdir(parents=True, exist_ok=True)
anchors_frame = metrics[metrics["is_train_anchor"]]
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(metrics["budget"], metrics["accuracy"], marker="o", label="all budgets")
ax.scatter(anchors_frame["budget"], anchors_frame["accuracy"], marker="*", s=130, color="crimson", label="training anchors")
ax.set(xlabel="Width budget", ylabel="Accuracy", title="CIFAR-100 accuracy vs width")
ax.grid(alpha=0.25); ax.legend(); fig.tight_layout()
fig.savefig(PLOT_DIR / "accuracy_vs_budget.png", dpi=180)
plt.show()

## 7. Local Wasserstein and accuracy sensitivity

The grid step is 0.05. `G(c)=SW(μc, μc+0.05)/0.05`; `P(c)=|Acc(c+0.05)-Acc(c)|/0.05`.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
axes[0].plot(local["budget_start"], local["G_width"], marker="o")
axes[0].set(xlabel="Interval start c", ylabel="G(c)", title="Local Sliced Wasserstein sensitivity")
axes[1].plot(local["budget_start"], local["P"], marker="o", color="darkorange")
axes[1].set(xlabel="Interval start c", ylabel="P(c)", title="Local accuracy sensitivity")
for axis in axes: axis.grid(alpha=0.25)
fig.tight_layout(); fig.savefig(PLOT_DIR / "local_sensitivities.png", dpi=180); plt.show()

## 8. Geometry vs performance and the mean-feature control

In [ ]:
from scipy.stats import pearsonr, spearmanr

pearson_result = pearsonr(local["G_width"], local["P"])
spearman_result = spearmanr(local["G_width"], local["P"])
control_row = control.loc[control["method"] == "euclidean_mean"].iloc[0]
control_rho = float(control_row["spearman_with_accuracy_sensitivity"])
control_p = float(control_row["spearman_p_value"])

fig, ax = plt.subplots(figsize=(6, 5))
points = ax.scatter(local["G_width"], local["P"], c=local["budget_start"], cmap="viridis", s=65)
ax.set(xlabel="Wasserstein sensitivity G(c)", ylabel="Accuracy sensitivity P(c)", title="Geometry vs performance")
ax.grid(alpha=0.25); fig.colorbar(points, ax=ax, label="Interval start c")
fig.tight_layout(); fig.savefig(PLOT_DIR / "geometry_vs_performance.png", dpi=180); plt.show()
print(f"Pearson r={pearson_result.statistic:.6f}, p={pearson_result.pvalue:.6g}")
print(f"Spearman rho={spearman_result.statistic:.6f}, p={spearman_result.pvalue:.6g}")
print(f"Euclidean-mean control: Spearman rho={control_rho:.6f}, p={control_p:.6g}")
difference = abs(float(spearman_result.statistic)) - abs(control_rho)
print(f"|rho_SW| - |rho_mean| = {difference:.6f}")
if difference <= 0:
    print("The Euclidean-mean control is equivalent or stronger in this run; Wasserstein has not shown incremental value.")

## 9. Pairwise Sliced Wasserstein heatmap

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))
image = ax.imshow(wasserstein, cmap="magma", origin="lower")
labels = [f"{budget:.2f}" for budget in metrics["budget"]]
ax.set_xticks(range(len(labels)), labels, rotation=90)
ax.set_yticks(range(len(labels)), labels)
ax.set(xlabel="Width budget", ylabel="Width budget", title="Pairwise Sliced Wasserstein distance")
fig.colorbar(image, ax=ax); fig.tight_layout()
fig.savefig(PLOT_DIR / "pairwise_wasserstein_heatmap.png", dpi=180)
plt.show()

## 10. GO / NO-GO summary and downloadable outputs

This is descriptive triage. The label is based on several visible signals and does not require correlation alone to cross a success threshold.

In [ ]:
geometry_values = local["G_width"].to_numpy(float)
performance_values = local["P"].to_numpy(float)
relative_geometry_range = float(np.ptp(geometry_values) / max(abs(np.mean(geometry_values)), 1e-12))
top_count = max(3, len(geometry_values) // 4)
top_geometry_indices = np.argsort(geometry_values)[-top_count:]
signals = {
    "geometry_not_nearly_flat": relative_geometry_range > 0.15,
    "high_geometry_tends_to_match_high_degradation": float(np.median(performance_values[top_geometry_indices])) > float(np.median(performance_values)),
    "notable_rank_association": np.isfinite(spearman_result.statistic) and abs(float(spearman_result.statistic)) >= 0.30,
    "compression_cliff_candidate_present": len(cliffs) > 0,
}
signal_count = sum(signals.values())
if signal_count >= 3:
    interpretation = "PROMISING SIGNAL"
elif signal_count >= 1:
    interpretation = "WEAK / INCONCLUSIVE SIGNAL"
else:
    interpretation = "NO OBVIOUS SIGNAL"

display(Markdown(f"## {interpretation}"))
for name, present in signals.items():
    print(f"- {name}: {present}")
print(f"- relative range of G(c): {relative_geometry_range:.6f}")
print(f"- compression cliff candidates: {len(cliffs)}")
if len(cliffs): display(cliffs)

In [ ]:
def format_number(value):
    return f"{float(value):.6g}" if np.isfinite(value) else "not estimable"

full_accuracy = float(metrics.loc[np.isclose(metrics["budget"], 1.0), "accuracy"].iloc[0])
quarter_accuracy = float(metrics.loc[np.isclose(metrics["budget"], 0.25), "accuracy"].iloc[0])
mean_seen = float(metrics.loc[metrics["is_train_anchor"], "accuracy"].mean())
mean_unseen = float(metrics.loc[~metrics["is_train_anchor"], "accuracy"].mean())
report_lines = [
    "# Kaggle Wasserstein geometry smoke-test summary",
    "",
    "- Dataset: CIFAR-100 (real data; FakeData disabled)",
    "- Backbone: slimmable CIFAR ResNet-18",
    f"- GPU: {torch.cuda.get_device_name(0)}",
    "- Seed: 0",
    "- Epochs: 20",
    "- Training anchors: 0.25, 0.50, 0.75, 1.00",
    "- Evaluation budgets: 0.25, 0.30, ..., 1.00",
    "",
    f"- Full-width accuracy: {full_accuracy:.6f}",
    f"- 25% width accuracy: {quarter_accuracy:.6f}",
    f"- Mean seen accuracy: {mean_seen:.6f}",
    f"- Mean unseen accuracy: {mean_unseen:.6f}",
    f"- Pearson G vs P: r={format_number(pearson_result.statistic)}, p={format_number(pearson_result.pvalue)}",
    f"- Spearman G vs P: rho={format_number(spearman_result.statistic)}, p={format_number(spearman_result.pvalue)}",
    f"- Euclidean-mean control: Spearman rho={format_number(control_rho)}, p={format_number(control_p)}",
    f"- Detected compression cliffs: {len(cliffs)}",
    f"- Interpretation: **{interpretation}**",
    "",
    "Evidence used by the descriptive multi-signal rule:",
    *[f"- {name}: {present}" for name, present in signals.items()],
    f"- relative range of G(c): {relative_geometry_range:.6f}",
    "",
    "> This is a one-seed, short-training smoke test intended only to assess whether the research hypothesis warrants a full experiment.",
    "",
    "No oracle, CFM, geometry regularization, adaptive width sampling, or hyperparameter search was run.",
]
KAGGLE_REPORT_PATH = RUN_DIR / "reports" / "kaggle_smoke_summary.md"
KAGGLE_REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
KAGGLE_REPORT_PATH.write_text("\n".join(report_lines) + "\n")
display(Markdown("\n".join(report_lines)))

In [ ]:
import shutil
from IPython.display import FileLink

archive_base = Path("/kaggle/working") / RUN_NAME
archive_path = Path(shutil.make_archive(str(archive_base), "zip", root_dir=RUN_DIR))
assert archive_path.is_file() and archive_path.stat().st_size > 0
print("All artifacts:", RUN_DIR)
print("Download archive:", archive_path)
display(FileLink(str(archive_path)))
print("Use Kaggle Save Version if this output must persist after the interactive session ends.")